# Compare Infomap and Leiden in a Scanpy workflow

Run Infomap on an AnnData neighbor graph and compare it with Scanpy's Leiden workflow. The example uses a small synthetic dataset, so it needs no downloads and stays reproducible.

Infomap reads the sparse observation graph from `adata.obsp` and writes categorical labels to `adata.obs`, matching Scanpy `tl` conventions. Leiden is the standard Scanpy baseline shown here.


In [ ]:
from importlib.metadata import version
import warnings

import infomap
import pandas as pd
from sklearn.datasets import make_blobs

# import scanpy after filterwarnings so its import-time warning stays silenced
warnings.filterwarnings("ignore", message="IProgress not found.*")
import scanpy as sc  # noqa: E402

print("infomap:", infomap.__version__)
print("scanpy:", version("scanpy"))
print("pandas:", pd.__version__)

## Create a small AnnData object

The dataset is synthetic and local. Scanpy builds a nearest-neighbor graph in `adata.obsp["connectivities"]`, which is the same graph used by Scanpy clustering tools and the default graph read by `infomap.tl.infomap()`.


In [ ]:
X, truth = make_blobs(
    n_samples=120,
    centers=4,
    n_features=12,
    cluster_std=1.8,
    random_state=123,
)
adata = sc.AnnData(X)
adata.obs["truth"] = pd.Categorical([str(label) for label in truth])

sc.pp.neighbors(adata, n_neighbors=12, random_state=123)
sc.tl.umap(adata, random_state=123)

adata

## Run Infomap and Leiden

`infomap.tl.infomap()` stores labels in `adata.obs[key_added]` and run metadata in `adata.uns[key_added]`. Scanpy's Leiden function is called with the same neighbor graph.


In [ ]:
infomap.tl.infomap(
    adata,
    key_added="infomap",
    seed=123,
    num_trials=20,
)

sc.tl.leiden(
    adata,
    key_added="leiden",
    random_state=123,
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

print("Infomap communities:", adata.obs["infomap"].nunique())
print("Leiden communities:", adata.obs["leiden"].nunique())

adata.uns["infomap"]

## Compare assignments

The labels are categorical strings because that is the common Scanpy representation for cluster assignments.


In [ ]:
comparison = adata.obs[["truth", "infomap", "leiden"]].copy()
comparison.head(10)

In [ ]:
from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score

metrics = pd.DataFrame(
    [
        {
            "method": "infomap",
            "AMI vs truth": adjusted_mutual_info_score(
                comparison["truth"], comparison["infomap"]
            ),
            "NMI vs truth": normalized_mutual_info_score(
                comparison["truth"], comparison["infomap"]
            ),
        },
        {
            "method": "leiden",
            "AMI vs truth": adjusted_mutual_info_score(
                comparison["truth"], comparison["leiden"]
            ),
            "NMI vs truth": normalized_mutual_info_score(
                comparison["truth"], comparison["leiden"]
            ),
        },
    ]
)
metrics

## Visualize the clusters

Color the same UMAP layout by the known synthetic labels and the detected communities.


In [ ]:
sc.pl.umap(adata, color=["truth", "infomap", "leiden"], wspace=0.35)

## Notes on graph choices

By default, Infomap uses `adata.obsp["connectivities"]`. Pass `neighbors_key` when using a named Scanpy neighbor graph, or `obsp`/`adjacency` when selecting a graph. Use `directed=True` for directed observation graphs and `use_weights=False` for unweighted treatment of nonzero sparse entries.

## Citation and further reading

If you use Infomap in published work, mapequation.org recommends citing two things: the map equation paper (Rosvall and Bergstrom, 2008, <https://doi.org/10.1073/pnas.0706851105>) for the method, and the MapEquation software package (Edler, Holmgren, and Rosvall) for the implementation and version. See [How to cite](https://www.mapequation.org/publications/) for BibTeX.

See also the {doc}`quickstart <quickstart>` for a first run and the {doc}`options guide <options-guide>` for the parameters used here.